# Load the SerpentGuard datasets

Start any EDA or training notebook from this one. It attaches the team's datasets by handle, so
nothing has to be added by hand, and loads any split with the same class ids everywhere.

- `serpentguard-m2-splits`: `splits.csv` (one row per photo, every label and split) and `classes.csv`.
  Small and quick, enough for most EDA.
- `serpentguard-indian-snakes-processed`: the photos in `data/<split>/<species>/`, plus `metadata.csv`
  and `classes.csv`.

The datasets are private. Ask the team to add your Kaggle username as a collaborator.

**Class ids:** always take `class_id` from `classes.csv`. `val/` and `test/` lack some species folders,
so loaders that number classes by folder (plain `ImageFolder`) would mislabel photos.


In [ ]:
from pathlib import Path

import kagglehub
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

SPLITS_HANDLE = "puneetbajajiitm/serpentguard-m2-splits"
PROCESSED_HANDLE = "puneetbajajiitm/serpentguard-indian-snakes-processed"

splits_root = Path(kagglehub.dataset_download(SPLITS_HANDLE))
splits = pd.read_csv(splits_root / "splits.csv", dtype={"obs_id": str, "photo_id": str, "observer_id": str})
print(len(splits), "photos;", splits["split"].value_counts().to_dict())


In [ ]:
processed_root = Path(kagglehub.dataset_download(PROCESSED_HANDLE))
meta = pd.read_csv(processed_root / "metadata.csv", dtype={"obs_id": str, "photo_id": str, "observer_id": str})
classes = pd.read_csv(processed_root / "classes.csv")
CLASS_ID = dict(zip(classes["species"], classes["class_id"]))
UNKNOWN_CLASS_ID = -1


class SerpentGuardSplit(Dataset):
    """One split of the processed dataset. Class ids come from classes.csv, so they match across splits."""

    def __init__(self, root, split, transform=None):
        rows = meta[meta["split"] == split]
        self.paths = [root / "data" / p for p in rows["path"]]
        self.labels = [CLASS_ID.get(s, UNKNOWN_CLASS_ID) for s in rows["species"]]
        self.extra = rows[["genus", "family", "who_category"]].to_dict("records")
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        image = Image.open(self.paths[i]).convert("RGB")
        return (self.transform(image) if self.transform else image), self.labels[i]


for split in ("train", "val", "test", "test_extra"):
    folders = sum(1 for p in (processed_root / "data" / split).iterdir() if p.is_dir())
    print(f"{split:10s} {len(SerpentGuardSplit(processed_root, split)):7d} photos, {folders:3d} of {len(CLASS_ID)} species folders")
classes.head(10)
